# RAG Pipeline — Fine-Tuned Qwen 2.5 Coder 7B

This notebook implements an **error-driven RAG pipeline** using a **fine-tuned** Qwen 2.5 Coder 7B model.

## Model
| Item | Value |
|------|-------|
| Fine-tuned model | `H4miid/qwen2_5_coder_7b_merged_f16.gguf` |
| Format | GGUF |
| Backend | llama.cpp server (OpenAI-compatible API) |

## Pipeline Overview
1. **Load Dataset** — train/test split
2. **Load Failed Samples** — from Pre-Test smoke report
3. **RAG Retrieval** — ChromaDB → BGE bi-encoder → cross-encoder reranking → **OpenRouter summarization (≤ 5 bullet hints)**
4. **LangChain Chain** — structured RAG prompt → llama.cpp server
5. **Evaluation** — syntax check + runtime smoke test

## Observability
- **LangSmith** traces every LLM call *and* every retrieval / reranking / summarization step.
```

## 1 — Imports

In [57]:
# --- Standard library ---
import json, os, re, time, subprocess, shutil, sys
import tempfile
from pathlib import Path
from difflib import SequenceMatcher
from datetime import datetime

# --- Data & ML ---
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# --- Vector DB & Embeddings ---
import chromadb
from sentence_transformers import SentenceTransformer, CrossEncoder

# --- LangChain & OpenAI ---
import openai, requests
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import OpenAI as LangchainOpenAI

# --- HuggingFace & Env ---
from huggingface_hub import hf_hub_download, login
from dotenv import load_dotenv

# --- LangSmith tracing ---
try:
    from langsmith import traceable
    LANGSMITH_OK = True
except ImportError:
    # If langsmith is not installed, create a no-op decorator so code still runs
    LANGSMITH_OK = False
    def traceable(*args, **_kwargs):
        """Dummy decorator when langsmith is not installed."""
        def decorator(fn):
            return fn
        if args and callable(args[0]):
            return args[0]
        return decorator

# ── Load .env ────────────────────────────────────────────────────────────────
load_dotenv()

# HuggingFace login for private model repos
HF_TOKEN = os.getenv("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN)
    print("✓ Logged in to HuggingFace Hub")
else:
    print("⚠ HF_TOKEN not found — private model download may fail")

# ── LangSmith setup (enable tracing if key exists) ───────────────────────────
LANGCHAIN_API_KEY = os.getenv("LANGCHAIN_API_KEY", "")
if LANGCHAIN_API_KEY:
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_API_KEY"]    = LANGCHAIN_API_KEY
    os.environ["LANGCHAIN_PROJECT"]    = "MentorApp-RAG-SFT"
    print("✓ LangSmith tracing enabled  →  project: MentorApp-RAG-SFT")
else:
    os.environ["LANGCHAIN_TRACING_V2"] = "false"
    print("ℹ LangSmith tracing disabled (add LANGCHAIN_API_KEY to .env to enable)")

print("✓ All imports loaded")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✓ Logged in to HuggingFace Hub
✓ LangSmith tracing enabled  →  project: MentorApp-RAG-SFT
✓ All imports loaded


## 0 — GPU Detection & llama.cpp Build

In [4]:
def check_gpu():
    """Return True if an NVIDIA GPU is available."""
    if shutil.which("nvidia-smi"):
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,memory.free",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=10,
        )
        if result.returncode == 0:
            for i, line in enumerate(result.stdout.strip().splitlines()):
                name, total, free = [p.strip() for p in line.split(",")]
                print(f"GPU {i}: {name}  |  VRAM: {int(total)/1024:.1f} GB total, {int(free)/1024:.1f} GB free")
            return True

    # Fallback: try PyTorch
    try:
        import torch
        if torch.cuda.is_available():
            print(f"GPU: {torch.cuda.get_device_name(0)}")
            return True
    except ImportError:
        pass

    print("⚠ No GPU detected — will run on CPU")
    return False

GPU_AVAILABLE = check_gpu()

GPU 0: NVIDIA GeForce RTX 5090  |  VRAM: 31.8 GB total, 31.4 GB free


In [5]:
# Clone llama.cpp and build it with CUDA support.
# This only needs to run once — subsequent runs skip the clone/build.

LLAMA_CPP_DIR = Path("../llama.cpp").resolve()

# Step 1: Clone the repo (skip if it already exists)
if not LLAMA_CPP_DIR.exists():
    print("Cloning llama.cpp …")
    subprocess.run(
        ["git", "clone", "https://github.com/ggerganov/llama.cpp.git", str(LLAMA_CPP_DIR)],
        check=True,
    )
    print(f"✓ Cloned → {LLAMA_CPP_DIR}")
else:
    print(f"✓ llama.cpp already at {LLAMA_CPP_DIR}")

# Step 2: CMake configure + build
build_dir = LLAMA_CPP_DIR / "build"
build_dir.mkdir(exist_ok=True)

print("Configuring with CUDA …")
subprocess.run(["cmake", "..", "-DGGML_CUDA=ON", "-DCMAKE_BUILD_TYPE=Release"],
               cwd=str(build_dir), check=True)

print("Building (may take a few minutes) …")
subprocess.run(["cmake", "--build", ".", "--config", "Release", "-j"],
               cwd=str(build_dir), check=True)

# Step 3: Locate the server executable
candidates = [
    build_dir / "bin" / "Release" / "llama-server.exe",  # Windows MSVC
    build_dir / "bin" / "llama-server.exe",               # Windows Ninja
    build_dir / "bin" / "llama-server",                   # Linux / macOS
]
LLAMA_SERVER_EXE = next((p for p in candidates if p.exists()), None)

# Fallback: recursive search
if not LLAMA_SERVER_EXE:
    hits = [f for f in build_dir.rglob("llama-server*") if f.is_file() and f.suffix in ("", ".exe")]
    if hits:
        LLAMA_SERVER_EXE = hits[0]

if LLAMA_SERVER_EXE:
    print(f"✓ Server executable: {LLAMA_SERVER_EXE}")
else:
    raise FileNotFoundError("llama-server not found after build — check CMake output.")

Cloning llama.cpp …


Cloning into '/workspace/MentorApp/llama.cpp'...


✓ Cloned → /workspace/MentorApp/llama.cpp
Configuring with CUDA …
-- The C compiler identification is GNU 13.3.0
-- The CXX compiler identification is GNU 13.3.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Found Git: /usr/bin/git (found version "2.43.0") 


CMAKE_BUILD_TYPE=Release


-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE  
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- Found OpenMP_C: -fopenmp (found version "4.5") 
-- Found OpenMP_CXX: -fopenmp (found version "4.5") 
-- Found OpenMP: TRUE (found version "4.5")  
-- x86 detected
-- Adding CPU backend variant ggml-cpu: -march=native 
-- Found CUDAToolkit: /usr/local/cuda/targets/x86_64-linux/include (found version "13.0.88") 
-- CUDA Toolkit found
-- The CUDA compiler identification is NVIDIA 13.0.88
-- Detecting CUDA compiler ABI info
-- Detecting CUDA compiler ABI info - done
-- Check for working CUDA compiler: /usr/local/cuda/bin/nvcc - skipped
-- Detecting CUDA compile features
-- Detecting CUDA compi

## 2 — Configuration

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_PATH       = Path("../Datasets/final_dataset.json").resolve()
CHROMA_DIR      = str(Path("../VectorDB/chroma_library_docs").resolve())
OUT_DIR         = Path("RAG_outputs/RAG_with_SFT").resolve()
MODEL_DIR       = Path("../Models").resolve()
COLLECTION_NAME = "library_docs"

SMOKE_REPORT_PATH = Path(
    r"../Fine-Tuning/Qwen/Fine-Tune_Results/fine_tuned_eval_runtime_A/smoke_report.json"
).resolve()

# Path to the SFT model's prediction outputs (contains prediction.py per sample)
SFT_EVAL_OUTPUTS_DIR = Path(
    "../Fine-Tuning/Qwen/Fine-Tune_Results/fine_tuned_eval_outputs_A/setting_A"
).resolve()

# ── Model ─────────────────────────────────────────────────────────────────────
HF_MODEL_REPO   = "H4miid/qwen2_5_coder_7b_merged_f16.gguf"
HF_MODEL_FILE   = "qwen2_5_coder_7b_merged_f16.gguf"
LOCAL_MODEL_PATH = MODEL_DIR / HF_MODEL_FILE

N_CTX        = 8192
N_GPU_LAYERS = -1 if GPU_AVAILABLE else 0   # -1 = offload everything to GPU
TEMPERATURE  = 0.0
MAX_TOKENS   = 4096

# ── Dataset split ─────────────────────────────────────────────────────────────
SEED      = 42
TEST_SIZE = 0.15

# ── RAG retrieval settings ───────────────────────────────────────────────────
N_RETRIEVE         = 10    # bi-encoder candidates per library
N_RERANK           = 3     # top-k kept after cross-encoder
MAX_QUERY          = 500   # max chars sent as RAG query
MAX_CTX_CHARS      = 3000  # char budget for raw context
MIN_RERANKER_SCORE = 0.0
RERANKER_MODEL     = "BAAI/bge-reranker-base"

# ── Runtime eval ──────────────────────────────────────────────────────────────
TIMEOUT   = 300
FORCE_CPU = False

# ── llama.cpp server ──────────────────────────────────────────────────────────
LLAMA_PORT     = 8081
LLAMA_HOST     = "127.0.0.1"
LLAMA_BASE_URL = f"http://{LLAMA_HOST}:{LLAMA_PORT}"

# ── OpenRouter (doc summarization) ───────────────────────────────────────────
OPENROUTER_API_KEY  = os.getenv("OPENROUTER_API_KEY", "")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_MODEL    = "openai/gpt-oss-20b:free"

# Fallback models — tried in order if the primary model returns None
OPENROUTER_FALLBACK_MODELS = [
    "openai/gpt-oss-120b:free",
    "mistralai/mistral-small-3.1-24b-instruct:free",
    "qwen/qwen3-4b:free",
]

# ── Summarization tuning ─────────────────────────────────────────────────────
SUMMARIZE_MAX_RETRIES   = 3        # retries per model before trying the next
SUMMARIZE_RETRY_DELAY   = 3        # initial delay in seconds (doubles each retry)
SUMMARIZE_THROTTLE      = 1.5      # minimum seconds between summarization API calls
SUMMARIZE_MAX_DOC_CHARS = 1500     # trim docs sent for summarization (less input → more reliable output)

# ── Create dirs ───────────────────────────────────────────────────────────────
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dataset         : {DATA_PATH}")
print(f"ChromaDB        : {CHROMA_DIR}")
print(f"Output dir      : {OUT_DIR}")
print(f"SFT eval outputs: {SFT_EVAL_OUTPUTS_DIR}")
print(f"Model           : {HF_MODEL_REPO}")
print(f"GPU layers      : {N_GPU_LAYERS}")
print(f"llama.cpp server: {LLAMA_BASE_URL}")
print(f"OpenRouter model: {OPENROUTER_MODEL}")
print(f"Fallback models : {OPENROUTER_FALLBACK_MODELS}")

Dataset         : /workspace/MentorApp/Datasets/final_dataset.json
ChromaDB        : /workspace/MentorApp/VectorDB/chroma_library_docs
Output dir      : /workspace/MentorApp/RAG_Pipelines/RAG_outputs/RAG_with_SFT
Model           : H4miid/qwen2_5_coder_7b_merged_f16.gguf
GPU layers      : -1
llama.cpp server: http://127.0.0.1:8081
OpenRouter model: openai/gpt-oss-20b:free
Fallback models : ['openai/gpt-oss-120b:free', 'mistralai/mistral-small-3.1-24b-instruct:free', 'qwen/qwen3-4b:free']


## 3 — Load Dataset & Split

In [59]:
# Load the full dataset
with open(DATA_PATH, "r", encoding="utf-8") as f:
    dataset_full = json.load(f)

print(f"Total samples: {len(dataset_full)}")

# Generate a synthetic ID for each sample from its index + title
def _make_sid(idx: int, title: str) -> str:
    """Create a short filesystem-safe identifier like '000_Adult_Income_Hyperparameter_Grid'."""
    slug = re.sub(r"[^A-Za-z0-9]+", "_", title).strip("_")
    return f"{idx:03d}_{slug}"

for i, sample in enumerate(dataset_full):
    sample["_sid"] = _make_sid(i, sample.get("title", f"sample_{i}"))
    sample["_orig_idx"] = i          # keep original 0-based index

# Train/test split
train_set, test_set = train_test_split(
    dataset_full,
    test_size=TEST_SIZE,
    random_state=SEED,
    shuffle=True,
)

print(f"Train set: {len(train_set)} samples")
print(f"Test set : {len(test_set)} samples")
print(f"Example SID: {dataset_full[0]['_sid']}")

Total samples: 582
Train set: 494 samples
Test set : 88 samples
Example SID: 000_Adult_Income_Hyperparameter_Grid


## 4 — Load Smoke Report & Identify Failed Samples

We only apply RAG to samples that **failed runtime** in the SFT pre-test. These are the samples the fine-tuned model struggled with and may benefit from retrieval augmentation.

In [ ]:
# Load the smoke report from SFT pre-test
with open(SMOKE_REPORT_PATH, "r", encoding="utf-8") as f:
    smoke_report = json.load(f)

print(f"Loaded smoke report with {len(smoke_report)} entries\n")

# NOTE: The smoke report was generated on Windows, so paths use backslashes.
# We use PureWindowsPath to parse them correctly on any OS.
from pathlib import PureWindowsPath

# ── Build mapping from original dataset index → smoke report entry ───────────
smoke_by_orig_idx = {}
for entry in smoke_report:
    test_pos = entry["idx"] - 1                       # 0-based position in test_set
    orig_idx = test_set[test_pos]["_orig_idx"]         # actual dataset index
    passed   = entry.get("ok", False) and not entry.get("skipped", False)

    # Extract the original folder name from the Windows file path
    eval_folder = PureWindowsPath(entry["file"]).parent.name

    smoke_by_orig_idx[orig_idx] = {**entry, "_passed": passed, "_eval_folder": eval_folder}

# Inject _eval_folder into the corresponding dataset samples
for orig_idx, info in smoke_by_orig_idx.items():
    dataset_full[orig_idx]["_eval_folder"] = info["_eval_folder"]

# ── Build rag_samples: failed samples with SFT prediction + runtime error ────
# For each failed sample we:
#   1. Read the SFT model's prediction.py from fine_tuned_eval_outputs_A  (= buggy code for RAG)
#   2. Use stderr_tail from the smoke report                             (= runtime error message)
#   3. Keep correct_code from the original dataset                       (= reference for similarity)

failed_entries = {
    orig_idx: info
    for orig_idx, info in smoke_by_orig_idx.items()
    if not info["_passed"]
}
print(f"Failed samples in pre-test: {len(failed_entries)}")

rag_samples = []
for orig_idx, info in sorted(failed_entries.items()):
    eval_folder = info["_eval_folder"]
    prediction_path = SFT_EVAL_OUTPUTS_DIR / eval_folder / "prediction.py"

    if not prediction_path.exists():
        print(f"  ⚠ SKIP {eval_folder}: prediction.py not found at {prediction_path}")
        continue

    sft_prediction = prediction_path.read_text(encoding="utf-8")
    stderr_error   = info.get("stderr_tail", "")
    ds_sample      = dataset_full[orig_idx]

    rag_samples.append({
        "_orig_idx":     orig_idx,
        "_eval_folder":  eval_folder,
        "_sid":          ds_sample["_sid"],
        "title":         ds_sample.get("title", ""),
        "correct_code":  ds_sample.get("correct_code", ""),
        # ── These come from the SFT eval, NOT the original dataset ────
        "sft_prediction": sft_prediction,       # the code the SFT model produced (failed)
        "runtime_error":  stderr_error,          # the runtime error from smoke test
    })

rag_samples.sort(key=lambda x: x["_orig_idx"])

print(f"\nSamples to process with RAG: {len(rag_samples)}")
for s in rag_samples[:10]:
    print(f"  - {s['_eval_folder']}")
if len(rag_samples) > 10:
    print(f"  ... and {len(rag_samples) - 10} more")

Loaded smoke report with 88 entries

Failed samples in pre-test: 18

Samples to process with RAG: 18
  - 046_Penguins_Migration_Time_Series_ARIMA
  - 080_IMDB_Sentiment_Bidirectional_LSTM_Embedd
  - 026_Titanic_Survival_ROC_Curve
  - 057_MNIST_Voting_Ensemble_Classification
  - 076_Fashion_MNIST_Transfer_Learning_with_Mob
  - 025_Diabetes_Progression_LightGBM_Regression
  - 071_Reuters_Topic_Classification_with_LSTM
  - 033_Digits_Autoencoder_Reconstruction
  - 017_Reuters_News_Topic_Classification
  - 083_Adult_Income_CatBoost_Classifier
  ... and 8 more


## 5 — Download Model & Start llama.cpp Server

Download the GGUF model from HuggingFace Hub, then start the **llama.cpp server** to serve it via an OpenAI-compatible API on `localhost`.

In [12]:
# Download the GGUF model from HuggingFace Hub if not already present
if not LOCAL_MODEL_PATH.exists():
    print(f"Downloading model from {HF_MODEL_REPO}...")
    downloaded_path = hf_hub_download(
        repo_id=HF_MODEL_REPO,
        filename=HF_MODEL_FILE,
        local_dir=str(MODEL_DIR),
        local_dir_use_symlinks=False
    )
    print(f"Downloaded to: {downloaded_path}")
else:
    print(f"Model already exists at: {LOCAL_MODEL_PATH}")

# Verify model file size
model_size_gb = LOCAL_MODEL_PATH.stat().st_size / (1024**3)
print(f"Model size: {model_size_gb:.2f} GB")

/workspace/MentorApp/.venv/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Downloaded to: /workspace/MentorApp/Models/qwen2_5_coder_7b_merged_f16.gguf
Model size: 14.19 GB


In [61]:
# Start the llama.cpp HTTP server (runs in the background).
# Then create a LangChain LLM client that talks to it.

print(f"Starting llama.cpp server on {LLAMA_BASE_URL} …")

server_process = subprocess.Popen(
    [str(LLAMA_SERVER_EXE),
     "-m",    str(LOCAL_MODEL_PATH),
     "--port", str(LLAMA_PORT),
     "--host", LLAMA_HOST,
     "-ngl",  str(N_GPU_LAYERS),
     "-c",    str(N_CTX)],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

# Wait until the server reports healthy 
for sec in range(120):
    try:
        if requests.get(f"{LLAMA_BASE_URL}/health", timeout=2).status_code == 200:
            print(f"✓ Server ready ({sec + 1}s)")
            break
    except requests.ConnectionError:
        pass
    time.sleep(1)
else:
    raise RuntimeError("llama.cpp server did not start within 120 s")

# LangChain LLM — points at the local server's OpenAI-compatible API
llm = LangchainOpenAI(
    base_url=f"{LLAMA_BASE_URL}/v1",
    api_key="not-needed",            # server has no auth
    model=LOCAL_MODEL_PATH.stem,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
)

# Smoke test
_ = llm.invoke("Say hello in one word.")
print("✓ LLM ready")

Starting llama.cpp server on http://127.0.0.1:8081 …
✓ Server ready (1s)
✓ LLM ready


## 6 — Load Vector Store & Reranker

Connect to the pre-built ChromaDB containing library documentation embeddings, and initialize the cross-encoder reranker for result refinement.

In [16]:
# Connect to ChromaDB
chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)
collection = chroma_client.get_collection(name=COLLECTION_NAME)

print(f"✓ Connected to ChromaDB collection: {COLLECTION_NAME}")
print(f"  Documents in collection: {collection.count()}")

# Initialize embedding model (same as used for indexing)
embedder = SentenceTransformer("BAAI/bge-base-en-v1.5")
print(f"✓ Embedding model loaded: BAAI/bge-base-en-v1.5")

# Initialize cross-encoder reranker
reranker = CrossEncoder(RERANKER_MODEL)
print(f"✓ Reranker loaded: {RERANKER_MODEL}")

✓ Connected to ChromaDB collection: library_docs
  Documents in collection: 647


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5512.52it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Embedding model loaded: BAAI/bge-base-en-v1.5


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6397.64it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Reranker loaded: BAAI/bge-reranker-base


## 7 — RAG Helper Functions

Three retrieval utilities (each traced by **LangSmith** when enabled):

| Function | What it does |
|----------|-------------|
| `retrieve_for_library` | Query ChromaDB for docs matching the error |
| `rerank_and_filter` | Cross-encoder reranking + score gate |
| `build_rag_context` | Orchestrate retrieval across libraries |

In [27]:
@traceable(name="retrieve_for_library")
def retrieve_for_library(query: str, library: str, n_results: int = N_RETRIEVE) -> list[dict]:
    """Search ChromaDB for docs about `library` that match `query`."""

    # Embed the query with the same model used during indexing
    query_vec = embedder.encode(query, normalize_embeddings=True).tolist()

    # Retrieve from ChromaDB, filtered by library name
    results = collection.query(
        query_embeddings=[query_vec],
        n_results=n_results,
        where={"library": library},
        include=["documents", "metadatas", "distances"],
    )

    # Package into simple dicts
    docs = []
    for i, text in enumerate(results["documents"][0]):
        docs.append({
            "text":     text,
            "metadata": results["metadatas"][0][i],
            "distance": results["distances"][0][i],
        })
    return docs


@traceable(name="rerank_and_filter")
def rerank_and_filter(query: str, docs: list[dict], top_k: int = N_RERANK) -> list[dict]:
    """Score each doc with the cross-encoder, keep the top_k above threshold."""

    if not docs:
        return []

    # Cross-encoder expects (query, document) pairs
    pairs  = [[query, d["text"]] for d in docs]
    scores = reranker.predict(pairs)

    for doc, score in zip(docs, scores):
        doc["rerank_score"] = float(score)

    # Sort descending and apply score gate
    ranked = sorted(docs, key=lambda d: d["rerank_score"], reverse=True)
    return [d for d in ranked if d["rerank_score"] >= MIN_RERANKER_SCORE][:top_k]


@traceable(name="build_rag_context")
def build_rag_context(error_text: str, libraries: list[str]) -> tuple[str, list[dict]]:
    """Retrieve + rerank across all libraries, return combined context string."""

    query    = error_text[:MAX_QUERY]
    all_docs = []

    for lib in libraries:
        raw_docs = retrieve_for_library(query, lib)
        reranked = rerank_and_filter(query, raw_docs)
        all_docs.extend(reranked)

    # Keep best-scoring docs first
    all_docs.sort(key=lambda d: d["rerank_score"], reverse=True)

    # Join texts up to the character budget
    parts, total = [], 0
    for doc in all_docs:
        if total + len(doc["text"]) > MAX_CTX_CHARS:
            break
        parts.append(doc["text"])
        total += len(doc["text"])

    context_str = "\n\n---\n\n".join(parts)
    return context_str, all_docs


# Quick test
_test = retrieve_for_library("SettingWithCopyWarning", "pandas", n_results=2)
print(f"Test: retrieved {len(_test)} docs for pandas")

Test: retrieved 2 docs for pandas


## 8 — Document Summarization (OpenRouter)

Use a free open-source model via **OpenRouter** to summarize retrieved & reranked documentation into **max 5 concise bullet-point hints** before appending them to the user prompt.

In [34]:
# OpenRouter client (used only for summarization — free tier)
# ⚠ OpenRouter REQUIRES HTTP-Referer for free models; without it responses are None.
openrouter_client = openai.OpenAI(
    base_url=OPENROUTER_BASE_URL,
    api_key=OPENROUTER_API_KEY,
    default_headers={
        "HTTP-Referer": "https://github.com/MentorApp",   # required by OpenRouter
        "X-Title":      "MentorApp-RAG",                   # recommended
    },
)
print(f"✓ OpenRouter client ready  ({OPENROUTER_MODEL})")

# Prompt that asks for ≤ 5 bullet hints
SUMMARIZE_PROMPT = """You are a concise technical assistant.
Summarize the following Python library documentation snippets into a
MAXIMUM of 5 bullet-point hints (use "•" as bullet character).
Each hint must help fix a Python runtime error.
Focus on: correct API usage, required parameters, common pitfalls.

Documentation:
{docs}

Respond with ONLY the bullet points. No intro, no conclusion."""

# Track the last API call time for throttling
_last_summarize_call = 0.0


def _local_bullet_fallback(docs_text: str, max_bullets: int = 5) -> str:
    """Last-resort fallback: extract informative sentences as bullet hints."""
    sentences = re.split(r'(?<=[.!?])\s+', docs_text.strip())
    keywords = re.compile(
        r"parameter|argument|return|raise|error|exception|deprecated|default|must|should|require|instead",
        re.IGNORECASE,
    )
    scored = []
    for s in sentences:
        s = s.strip()
        if len(s) < 20 or len(s) > 300:
            continue
        scored.append((len(keywords.findall(s)), s))

    scored.sort(key=lambda x: x[0], reverse=True)
    seen, bullets = set(), []
    for _, s in scored:
        if s not in seen:
            seen.add(s)
            bullets.append(f"• {s}")
        if len(bullets) >= max_bullets:
            break
    if not bullets:
        for s in sentences[:max_bullets]:
            s = s.strip()
            if s:
                bullets.append(f"• {s}")
    return "\n".join(bullets)


def _try_summarize_with_model(model: str, prompt: str) -> str | None:
    """Call a single OpenRouter model. Returns bullet string or None."""
    global _last_summarize_call

    # ── Throttle: respect minimum gap between API calls ───────────────────
    elapsed = time.time() - _last_summarize_call
    if elapsed < SUMMARIZE_THROTTLE:
        time.sleep(SUMMARIZE_THROTTLE - elapsed)

    delay = SUMMARIZE_RETRY_DELAY

    for attempt in range(1, SUMMARIZE_MAX_RETRIES + 1):
        try:
            _last_summarize_call = time.time()
            resp = openrouter_client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": "You summarize documentation into concise bullet points."},
                    {"role": "user",   "content": prompt},
                ],
                max_tokens=400,
                temperature=0.0,
            )

            # ── Diagnostic: inspect the raw response ─────────────────────
            choice = resp.choices[0] if resp.choices else None
            if choice is None:
                print(f"    [WARN] {model}: resp.choices is empty (attempt {attempt})")
                if attempt < SUMMARIZE_MAX_RETRIES:
                    time.sleep(delay); delay *= 2
                    continue
                return None

            content = choice.message.content

            # Some models put output in 'reasoning_content' instead
            if content is None and hasattr(choice.message, "reasoning_content"):
                content = getattr(choice.message, "reasoning_content", None)

            if content is None:
                finish = getattr(choice, "finish_reason", "unknown")
                print(f"    [WARN] {model} returned None content "
                      f"(finish_reason={finish}, attempt {attempt}/{SUMMARIZE_MAX_RETRIES})")
                if attempt < SUMMARIZE_MAX_RETRIES:
                    time.sleep(delay); delay *= 2
                    continue
                return None

            summary = content.strip()
            # Accept bullets with "•", "-", or "*"
            bullets = [
                l.strip() for l in summary.split("\n")
                if l.strip() and l.strip()[0] in ("•", "-", "*")
            ]
            if bullets:
                normalized = [("• " + b.lstrip("•-* ")) for b in bullets[:5]]
                return "\n".join(normalized)
            # Model responded but without bullet format — still usable
            return summary[:600]

        except Exception as e:
            print(f"    [WARN] {model} error (attempt {attempt}/{SUMMARIZE_MAX_RETRIES}): {e}")
            if attempt < SUMMARIZE_MAX_RETRIES:
                time.sleep(delay); delay *= 2
            else:
                return None

    return None


@traceable(name="summarize_rag_docs")
def summarize_rag_docs(docs_text: str) -> str:
    """Summarize retrieved docs into ≤ 5 bullet hints via OpenRouter.

    Strategy to maximize LLM-quality bullets:
    1. Trim docs to SUMMARIZE_MAX_DOC_CHARS (less input → more reliable output)
    2. Throttle calls (SUMMARIZE_THROTTLE seconds gap) to avoid rate limits
    3. Retry with exponential backoff on None responses
    4. If primary model fails, try each OPENROUTER_FALLBACK_MODELS in order
    5. Local sentence extraction only as absolute last resort
    """
    if not docs_text.strip():
        return ""

    prompt = SUMMARIZE_PROMPT.format(docs=docs_text[:SUMMARIZE_MAX_DOC_CHARS])

    # Try primary model first, then fallbacks
    models_to_try = [OPENROUTER_MODEL] + OPENROUTER_FALLBACK_MODELS

    for model in models_to_try:
        result = _try_summarize_with_model(model, prompt)
        if result is not None:
            return result
        print(f"  [INFO] {model} exhausted — trying next model …")

    # All models failed → local extraction as last resort
    print("  [INFO] All OpenRouter models failed — using local bullet extraction")
    return _local_bullet_fallback(docs_text)


@traceable(name="build_rag_context_with_summary")
def build_rag_context_with_summary(
    error_text: str,
    libraries: list[str],
    summarize: bool = True,
) -> tuple[str, str, list[dict]]:
    """Retrieve → rerank → (optionally) summarize into bullet hints."""

    raw_context, all_docs = build_rag_context(error_text, libraries)

    if not raw_context:
        return "", "", all_docs

    if summarize:
        print("  Summarizing via OpenRouter …")
        summarized = summarize_rag_docs(raw_context)
    else:
        summarized = raw_context

    return raw_context, summarized, all_docs


print("✓ Summarization functions defined")

✓ OpenRouter client ready  (openai/gpt-oss-20b:free)
✓ Summarization functions defined


## 9 — LangChain Chain Setup

Create a `PromptTemplate` and `LLMChain` connected to the **llama.cpp server** for RAG-augmented code generation.

**Components:**
- **System prompt**: Defines the model's role as a Python bug-fixer with structured output format
- **User prompt**: Contains buggy code, error message, and summarized bullet-point hints from OpenRouter
- **Output format**: Uses `<correct_code>` XML tags for reliable parsing

In [35]:
# ── System prompt (tells the model how to respond) ───────────────────────────
SYSTEM_PROMPT = (
    "You fix Python programs.\n"
    "Input: a traceback tail, buggy Python code, and sometimes hints.\n"
    "Use the hints when helpful and fix the Python code.\n"
    "Return ONLY this format:\n"
    "<correct_code>\n"
    "(full corrected python code)\n"
    "</correct_code>\n"
    "<error_type>\n"
    "(one short line describing the original bug type)\n"
    "</error_type>\n"
    "No markdown, no backticks, no explanations, do not echo the prompt."
)


# ── User prompt template (filled per sample) ─────────────────────────────────
USER_PROMPT_TEMPLATE = """Fix this Python code based on the runtime error.

Traceback:
{error_message}

Code:
{buggy_code}

Reference docs:
{context}"""

# ── Combine into a single prompt template for LangChain ──────────────────────
RAG_PROMPT_TEMPLATE = f"""{SYSTEM_PROMPT}

{{user_prompt}}"""

prompt_template = PromptTemplate(
    input_variables=["user_prompt"],
    template=RAG_PROMPT_TEMPLATE,
)

rag_chain = prompt_template | llm | StrOutputParser()

print("System prompt:")
print(SYSTEM_PROMPT)
print("\n✓ LangChain chain ready")

System prompt:
You fix Python programs.
Input: a traceback tail, buggy Python code, and sometimes hints.
Use the hints when helpful and fix the Python code.
Return ONLY this format:
<correct_code>
(full corrected python code)
</correct_code>
<error_type>
(one short line describing the original bug type)
</error_type>
No markdown, no backticks, no explanations, do not echo the prompt.

✓ LangChain chain ready


## 10 — Output Parsing & Similarity

Helper functions to extract Python code from model output and compute similarity to reference solutions.

**Parsing priority:**
1. `<correct_code>` XML tags (preferred format)
2. Markdown code blocks (```python)
3. Raw text fallback

In [36]:
def extract_python_code(text: str) -> str:
    """Pull code from model output: <correct_code> tags → ```python block → raw."""
    text = (text or "").strip()

    # Try <correct_code> tags first
    m = re.search(r"<correct_code>\s*(.*?)\s*</correct_code>", text, re.DOTALL | re.IGNORECASE)
    if m:
        code = m.group(1).strip()
        code = re.sub(r"^```\w*\s*", "", code)    # strip opening fence
        code = re.sub(r"\s*```$", "", code)        # strip closing fence
        return code.strip()

    # Try markdown code blocks
    blocks = re.findall(r"```(?:python)?\s*([\s\S]*?)```", text, re.IGNORECASE)
    if blocks:
        return blocks[-1].strip()

    return text.strip()


def extract_error_type(text: str) -> str:
    """Pull error type from <error_type> tags."""
    m = re.search(r"<error_type>\s*(.*?)\s*</error_type>", (text or ""), re.DOTALL | re.IGNORECASE)
    return m.group(1).strip() if m else ""


def calculate_similarity(code1: str, code2: str) -> float:
    """Character-level similarity between two code strings (0–1)."""
    if not code1 or not code2:
        return 0.0
    return SequenceMatcher(None, code1, code2).ratio()


def get_libraries_from_sample(sample: dict) -> list[str]:
    """Return library names from sample metadata, or infer from imports."""
    if "libraries" in sample:
        libs = sample["libraries"]
        return [libs] if isinstance(libs, str) else list(libs)

    # Infer from the buggy code (field is 'incorrect_code' in this dataset)
    code = sample.get("incorrect_code", "")
    known = ["pandas", "numpy", "sklearn", "matplotlib", "tensorflow", "keras", "torch"]
    found = [lib for lib in known if lib in code]
    return found or ["pandas"]


# Quick test
_sample_out = "<correct_code>\nimport pandas as pd\nprint('ok')\n</correct_code>"
print("Extraction test:", extract_python_code(_sample_out)[:60])

Extraction test: import pandas as pd
print('ok')


## 11 — RAG Inference Loop

Process each failed sample through the RAG pipeline:
1. Extract libraries from sample
2. Build RAG context from error message (retrieve + rerank)
3. **Summarize** reranked docs into ≤ 5 bullet hints via OpenRouter
4. Run the LangChain chain via llama.cpp server
5. Extract corrected code from `<correct_code>` tags

In [ ]:
results = []
total   = len(rag_samples)

print(f"Processing {total} samples …\n" + "=" * 70)

for idx, sample in enumerate(rag_samples):
    sid          = sample["_sid"]
    eval_folder  = sample["_eval_folder"]
    print(f"\n[{idx+1}/{total}] {eval_folder}")

    # ── The "buggy code" is the SFT model's failed prediction ─────────────
    buggy_code    = sample["sft_prediction"]
    # ── The "error message" is the stderr from the smoke test ─────────────
    error_message = sample["runtime_error"]
    # ── Reference correct code from the original dataset ──────────────────
    correct_code  = sample.get("correct_code", "")
    task          = sample.get("title", "")
    libraries     = get_libraries_from_sample({"incorrect_code": buggy_code})

    # ── RAG: retrieve → rerank → summarize ────────────────────────────────
    raw_ctx, summary_ctx, docs = build_rag_context_with_summary(
        error_message, libraries, summarize=True,
    )
    print(f"  docs={len(docs)}  raw={len(raw_ctx)}ch  summary={len(summary_ctx)}ch")

    # ── Build prompt & call LLM ───────────────────────────────────────────
    user_prompt = USER_PROMPT_TEMPLATE.format(
        buggy_code=buggy_code,
        error_message=error_message,
        context=summary_ctx or "No relevant documentation found.",
    )

    # Full prompt (system + user) as sent to the coder model
    full_input_prompt = RAG_PROMPT_TEMPLATE.format(user_prompt=user_prompt)

    try:
        raw_output = rag_chain.invoke({"user_prompt": user_prompt})
    except Exception as e:
        print(f"  ERROR: {e}")
        raw_output = ""

    predicted   = extract_python_code(raw_output)
    error_type  = extract_error_type(raw_output)
    sim         = calculate_similarity(predicted, correct_code)
    print(f"  similarity={sim:.3f}" + (f"  error_type={error_type}" if error_type else ""))

    # ── Save ──────────────────────────────────────────────────────────────
    result = dict(
        sid=sid, eval_folder=eval_folder, title=task, libraries=libraries,
        buggy_code=buggy_code, error_message=error_message, correct_code=correct_code,
        raw_rag_context=raw_ctx, summarized_context=summary_ctx,
        raw_output=raw_output, predicted_code=predicted,
        error_type_detected=error_type, sim_to_ref=sim, n_docs_retrieved=len(docs),
        _orig_idx=sample["_orig_idx"],
    )
    results.append(result)

    out_dir = OUT_DIR / eval_folder
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "predicted.py").write_text(predicted, encoding="utf-8")
    (out_dir / "input_prompt.txt").write_text(full_input_prompt, encoding="utf-8")
    (out_dir / "raw_output.txt").write_text(raw_output, encoding="utf-8")
    meta = {k: v for k, v in result.items() if k not in ("predicted_code", "buggy_code", "correct_code")}
    (out_dir / "metadata.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

print("\n" + "=" * 70)
print(f"✓ Done — {len(results)} samples processed")

Processing 18 samples …

[1/18] 046_Penguins_Migration_Time_Series_ARIMA
  Summarizing via OpenRouter …
  docs=6  raw=2960ch  summary=739ch
  similarity=1.000  error_type=NameError

[2/18] 080_IMDB_Sentiment_Bidirectional_LSTM_Embedd
  Summarizing via OpenRouter …
  docs=9  raw=2914ch  summary=87ch
  similarity=0.054

[3/18] 026_Titanic_Survival_ROC_Curve
  Summarizing via OpenRouter …
    [WARN] openai/gpt-oss-20b:free error (attempt 1/3): Error code: 429 - {'error': {'message': 'Provider returned error', 'code': 429, 'metadata': {'raw': 'openai/gpt-oss-20b:free is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations', 'provider_name': 'OpenInference', 'is_byok': False}}, 'user_id': 'user_34T64NRW09AE1IOhHpOs4qHu0F8'}
  docs=6  raw=2730ch  summary=835ch
  similarity=0.997  error_type=LogicError

✓ Done — 3 samples processed


## 12 — Save Summary Results

Export all results to a JSON file for analysis.

In [ ]:
summary_path = OUT_DIR / "summary_rag_sft.json"

summary_data = {
    "model": HF_MODEL_REPO,
    "total_samples": len(results),
    "timestamp": datetime.now().isoformat(),
    "config": dict(n_ctx=N_CTX, temperature=TEMPERATURE, max_tokens=MAX_TOKENS,
                   n_retrieve=N_RETRIEVE, n_rerank=N_RERANK,
                   min_reranker_score=MIN_RERANKER_SCORE, max_ctx_chars=MAX_CTX_CHARS),
    "results": results,
}

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary_data, f, indent=2)

print(f"✓ Saved {len(results)} results → {summary_path}")

## 13 — Results Summary

Display statistics about the RAG inference results.

In [ ]:
sims      = [r["sim_to_ref"] for r in results]
parseable = [r for r in results if r["predicted_code"].strip()]

print("=" * 50)
print("RAG INFERENCE SUMMARY")
print("=" * 50)
print(f"Samples   : {len(results)}")
print(f"Parseable : {len(parseable)}")

if sims:
    print(f"Similarity: mean={sum(sims)/len(sims):.3f}  min={min(sims):.3f}  max={max(sims):.3f}")
    high = sum(s >= 0.8 for s in sims)
    med  = sum(0.5 <= s < 0.8 for s in sims)
    low  = sum(s < 0.5 for s in sims)
    print(f"  ≥0.8: {high}   0.5–0.8: {med}   <0.5: {low}")

---

## 14 — Evaluation: Syntax Check

Verify that generated code compiles without syntax errors using `py_compile`.

In [ ]:
def check_syntax(code: str) -> tuple[bool, str]:
    """Return (True, '') if code compiles, else (False, error_msg)."""
    if not code.strip():
        return False, "Empty code"
    try:
        compile(code, "<string>", "exec")
        return True, ""
    except SyntaxError as e:
        return False, f"Line {e.lineno}: {e.msg}"

syntax_results = []
for r in results:
    ok, err = check_syntax(r["predicted_code"])
    syntax_results.append({"sid": r["sid"], "valid": ok, "error": err})

valid_count = sum(s["valid"] for s in syntax_results)
print(f"Syntax valid: {valid_count}/{len(syntax_results)}")

for s in syntax_results:
    if not s["valid"]:
        print(f"  ✗ {s['sid']}: {s['error']}")

## 15 — Fast Eval Patching

Reduce epochs and iterations for faster runtime testing.

In [ ]:
NO_EPOCH_PATCH = {"081"}   # samples that must keep original epoch count

def patch_fast_eval(code: str, skip_epoch_patch: bool = False) -> str:
    """Reduce epochs / verbosity so smoke tests finish quickly."""
    if not skip_epoch_patch:
        code = re.sub(r'epochs\s*=\s*\d+',   'epochs=5',    code)
        code = re.sub(r'n_iter\s*=\s*\d+',   'n_iter=5',    code)
        code = re.sub(r'max_iter\s*=\s*\d+', 'max_iter=100', code)

    code = re.sub(r'verbose\s*=\s*[12]', 'verbose=0', code)

    if FORCE_CPU and "CUDA_VISIBLE_DEVICES" not in code:
        code = 'import os\nos.environ["CUDA_VISIBLE_DEVICES"] = ""\n' + code
    return code

# Quick demo
print(patch_fast_eval("model.fit(X, y, epochs=100, verbose=1)"))

## 16 — Runtime Smoke Test

Run each generated script in an isolated subprocess to check for runtime errors.

In [ ]:
def run_script(code: str, timeout: int = TIMEOUT) -> dict:
    """Run code in a subprocess, return status + stdout/stderr + runtime."""
    with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False, encoding="utf-8") as f:
        f.write(code)
        tmp = f.name
    start = time.time()
    try:
        r = subprocess.run([sys.executable, tmp],
                           capture_output=True, text=True,
                           timeout=timeout, cwd=str(OUT_DIR))
        dt = time.time() - start
        return {"status": "pass" if r.returncode == 0 else "runtime_error",
                "stdout": r.stdout[:2000], "stderr": r.stderr[:2000], "runtime": dt}
    except subprocess.TimeoutExpired:
        return {"status": "timeout", "stdout": "", "stderr": f"Timeout {timeout}s", "runtime": timeout}
    except Exception as e:
        return {"status": "error", "stdout": "", "stderr": str(e), "runtime": time.time() - start}
    finally:
        try: os.unlink(tmp)
        except OSError: pass


# Run smoke tests
valid_samples = [r for r, s in zip(results, syntax_results) if s["valid"]]
smoke_results = []

print(f"Smoke-testing {len(valid_samples)} samples …\n" + "=" * 60)

for i, r in enumerate(valid_samples):
    sid = r["sid"]
    idx_str = sid.split("_")[0]           # e.g. "081"
    patched = patch_fast_eval(r["predicted_code"], skip_epoch_patch=(idx_str in NO_EPOCH_PATCH))

    res = run_script(patched)
    smoke_results.append({"sid": sid, "_orig_idx": r["_orig_idx"], **res})

    icon = "✓" if res["status"] == "pass" else "✗"
    print(f"[{i+1}/{len(valid_samples)}] {icon} {sid}  {res['status']}  ({res['runtime']:.1f}s)")
    if res["status"] != "pass" and res["stderr"]:
        last_line = res["stderr"].strip().split("\n")[-1]
        print(f"        {last_line[:100]}")

print("=" * 60)

## 17 — Evaluation Summary

Aggregate evaluation results and save smoke test report.

In [ ]:
from collections import Counter

counts = Counter(sr["status"] for sr in smoke_results)
passed = counts.get("pass", 0)
total  = len(smoke_results)

print("=" * 50)
print("SMOKE TEST SUMMARY")
print("=" * 50)
for status, n in counts.most_common():
    print(f"  {status}: {n}  ({100*n/total:.0f}%)")
print(f"\nPass rate: {passed}/{total} ({100*passed/total:.0f}%)")

# Save report
smoke_out = OUT_DIR / "smoke_report_rag_sft.json"
with open(smoke_out, "w") as f:
    json.dump(smoke_results, f, indent=2)
print(f"✓ Report saved → {smoke_out}")

## 18 — Comparison: SFT vs SFT+RAG

Compare the RAG-enhanced results with the original SFT model pre-test results.

In [ ]:
# Compare original SFT pre-test vs SFT + RAG
# smoke_by_orig_idx was built in Cell 12 with _passed flag

total_samples = len(dataset_full)
orig_passed   = sum(1 for info in smoke_by_orig_idx.values() if info["_passed"])
orig_failed   = total_samples - orig_passed

rag_fixed = sum(
    1 for sr in smoke_results
    if not smoke_by_orig_idx.get(sr["_orig_idx"], {}).get("_passed", True)
    and sr["status"] == "pass"
)
still_failed = sum(
    1 for sr in smoke_results
    if not smoke_by_orig_idx.get(sr["_orig_idx"], {}).get("_passed", True)
    and sr["status"] != "pass"
)
final_passed = orig_passed + rag_fixed

print("=" * 60)
print("SFT  vs  SFT + RAG")
print("=" * 60)
print(f"Dataset size       : {total_samples}")
print(f"SFT passed         : {orig_passed}  ({100*orig_passed/total_samples:.1f}%)")
print(f"SFT failed         : {orig_failed}")
print(f"RAG fixed          : {rag_fixed}")
print(f"Still failed       : {still_failed}")
print(f"Final pass rate    : {final_passed}/{total_samples}  ({100*final_passed/total_samples:.1f}%)")
delta = 100 * final_passed / total_samples - 100 * orig_passed / total_samples
print(f"Improvement        : +{delta:.1f} pp")
print("=" * 60)